### Part 1
#### 1.2 
Next, fine-tune this model for this task.

In [1]:
# # Lab 3.3 Part 1.2: Fine-tuning BERT for fMRI voxel prediction

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer, BertModel
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ## Load Raw Text Data
raw_text_path = Path("/ocean/projects/mth240012p/shared/data/raw_text.pkl")
with open(raw_text_path, "rb") as f:
    raw_text = pickle.load(f)

# ## Load fMRI Data
from tqdm import tqdm

def load_subject_data(subject_dir):
    file_list = list(subject_dir.glob("*.npy"))
    print(f" Found {len(file_list)} files in {subject_dir}")
    
    data = {}
    for f in tqdm(file_list, desc=f"Loading {subject_dir.name}"):
        try:
            data[f.stem] = np.load(f)
        except Exception as e:
            print(f" Failed to load {f.name}: {e}")
    return data
    
subject2_dir = Path("/jet/home/xzhan/tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject2")
subject3_dir = Path("/jet/home/xzhan/tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject3")

Y_subject2 = load_subject_data(subject2_dir)
Y_subject3 = load_subject_data(subject3_dir)

/jet/home/xzhan/.conda/envs/env_214/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/tmp/ipykernel_1844/4204634595.py:19: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  raw_text = pickle.load(f)


Using device: cuda
 Found 101 files in /jet/home/xzhan/tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject2


Loading subject2: 100%|██████████| 101/101 [00:08<00:00, 11.85it/s]


 Found 101 files in /jet/home/xzhan/tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject3


Loading subject3: 100%|██████████| 101/101 [00:08<00:00, 12.30it/s]


In [13]:
print(f"raw_text type: {type(raw_text)}")
sample_key = list(raw_text.keys())[0]
print(f"Sample story ID: {sample_key}")
print(f"raw_text[{sample_key}] type: {type(raw_text[sample_key])}")
print(f"raw_text[{sample_key}].data[:5]: {raw_text[sample_key].data[:5]}")

raw_text type: <class 'dict'>
Sample story ID: sweetaspie
raw_text[sweetaspie] type: <class 'ridge_utils.DataSequence.DataSequence'>
raw_text[sweetaspie].data[:5]: ['', 'i', 'embarked', 'on', 'a']


In [16]:
print(f"Y_subject2 type: {type(Y_subject2)}")
print(f"Y_subject2 keys: {list(Y_subject2.keys())[:3]}")
sid = list(Y_subject2.keys())[0]
print(f"Y_subject2[{sid}].shape: {Y_subject2[sid].shape}")
print(f"Sample voxel vector: {Y_subject2[sid][0][:5]}")

Y_subject2 type: <class 'dict'>
Y_subject2 keys: ['stumblinginthedark', 'singlewomanseekingmanwich', 'theclosetthatateeverything']
Y_subject2[stumblinginthedark].shape: (489, 94251)
Sample voxel vector: [ 0.27800206 -1.4666673   0.17739297  0.62328273  0.30665753]


In [18]:
from itertools import islice

for sid in islice((sid for sid in Y_subject2 if sid in raw_text), 10):
    print(f"{sid} — raw_text: {len(raw_text[sid].data)}  |  Y: {Y_subject2[sid].shape[0]}")

stumblinginthedark — raw_text: 2681  |  Y: 489
singlewomanseekingmanwich — raw_text: 1486  |  Y: 297
theclosetthatateeverything — raw_text: 1928  |  Y: 314
jugglingandjesus — raw_text: 887  |  Y: 193
threemonths — raw_text: 2062  |  Y: 353
escapingfromadirediagnosis — raw_text: 1423  |  Y: 343
wildwomenanddancingqueens — raw_text: 1218  |  Y: 189
igrewupinthewestborobaptistchurch — raw_text: 2449  |  Y: 439
undertheinfluence — raw_text: 1641  |  Y: 304
quietfire — raw_text: 1905  |  Y: 455


In [20]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertModel, BertTokenizer
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from preprocessing import downsample_word_vectors, make_delayed
from ridge_utils.ridge import ridge_corr_pred

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DelayedBERTDataset(Dataset):
    def __init__(self, raw_text, fmri_data, delays=range(1, 5), TR=1.0, skip_seconds=(5, 10), window_size=8):
        self.samples = []

        word_vectors = {}
        for sid in raw_text:
            words = raw_text[sid].data
            word_vectors[sid] = np.zeros((len(words), 1), dtype=np.float32)  # 注意：只用 1 维即可


        print("Downsampling word sequences to TR resolution...")
        downsampled = downsample_word_vectors(
            stories=list(raw_text.keys()),
            word_vectors=word_vectors,
            wordseqs=raw_text
        )

        trim_start = int(skip_seconds[0] / TR)
        trim_end = int(skip_seconds[1] / TR)
        trimmed = {
            sid: X[trim_start:-trim_end] for sid, X in downsampled.items()
            if X.shape[0] > (trim_start + trim_end)
        }

        print("Creating delayed input features...")
        delayed = {sid: make_delayed(X, delays=delays) for sid, X in trimmed.items()}

        print("Pairing TRs with token windows...")
        for sid in delayed:
            if sid not in fmri_data or sid not in raw_text:
                continue

            Y = fmri_data[sid]
            words = raw_text[sid].data
            T = len(delayed[sid])
            n = min(T, len(Y))
            for i in range(n):
                y = Y[i]
                if np.isnan(y).any():
                    continue
                # Approximate alignment between TR index and word index
                word_idx = int(i * len(words) / len(Y))
                left = max(0, word_idx - window_size)
                right = min(len(words), word_idx + window_size)
                word_window = words[left:right]
                self.samples.append((word_window, y))

        print(f"Total usable samples: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        word_window, y = self.samples[idx]
        text_input = " ".join(word_window)
        return {
            "input": text_input,
            "target": torch.tensor(y, dtype=torch.float)
        }
        
        
# === Fine-tunable BERT model ===
class BERTVoxelRegressor(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.linear = nn.Linear(self.bert.config.hidden_size, output_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.linear(cls)

# === Training pipeline ===
def train_finetune_bert(raw_text, fmri_data, subject_name):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    dataset = DelayedBERTDataset(raw_text, fmri_data)
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])
    train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32)
    test_loader = DataLoader(test_set, batch_size=32)

    output_dim = dataset[0]["target"].shape[0]
    model = BERTVoxelRegressor(output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience = 3
    pat_counter = 0

    output_dir = f"output/fine_tune/{subject_name}"
    os.makedirs(output_dir, exist_ok=True)
    best_model_path = os.path.join(output_dir, "bert_finetuned.pth")

    for epoch in range(20):
        model.train()
        train_loss = 0
        for batch in train_loader:
            inputs = tokenizer(batch["input"], padding=True, truncation=True,
                               return_tensors="pt", max_length=16).to(device)
            targets = batch["target"].to(device)
            optimizer.zero_grad()
            preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = tokenizer(batch["input"], padding=True, truncation=True,
                                   return_tensors="pt", max_length=16).to(device)
                targets = batch["target"].to(device)
                preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
                val_loss += criterion(preds, targets).item()

        print(f"[{subject_name}] Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            pat_counter = 0
        else:
            pat_counter += 1
            if pat_counter >= patience:
                print(" Early stopping")
                break

    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs = tokenizer(batch["input"], padding=True, truncation=True,
                               return_tensors="pt", max_length=16).to(device)
            targets = batch["target"].cpu().numpy()
            preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets)

    pred = np.concatenate(all_preds, axis=0)
    true = np.concatenate(all_targets, axis=0)
    Rstim = pred
    Pstim = pred
    Rresp = true
    Presp = true
    valphas = np.ones(Rresp.shape[1])
    ccs = ridge_corr_pred(Rstim, Pstim, Rresp, Presp, valphas)

    np.save(os.path.join(output_dir, "voxel_cc.npy"), ccs)
    with open(os.path.join(output_dir, "summary_cc.txt"), "w") as f:
        f.write(f"Subject: {subject_name}\\n")
        f.write(f"Mean CC: {np.mean(ccs):.4f}\\n")
        f.write(f"Median CC: {np.median(ccs):.4f}\\n")
        f.write(f"Top 5%: {np.quantile(ccs, 0.95):.4f}\\n")
        f.write(f"Top 1%: {np.quantile(ccs, 0.99):.4f}\\n")

    plt.figure()
    plt.hist(ccs, bins=50, color='skyblue', edgecolor='black')
    plt.title(f"CC Distribution for {subject_name}")
    plt.xlabel("Voxel-wise Correlation (CC)")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "cc_distribution.png"), dpi=300)
    plt.close()
    print(f"Final mean CC: {np.mean(ccs):.4f}")
    return ccs

# === Run for both subjects ===
subject_map = {
    "subject2": Y_subject2,
    "subject3": Y_subject3
}

all_ccs = {}
for name, ydata in subject_map.items():
    all_ccs[name] = train_finetune_bert(raw_text, ydata, name)

for name, ccs in all_ccs.items():
    print(f"== {name} Summary ==")
    print(f" Mean: {np.mean(ccs):.4f}")
    print(f" Median: {np.median(ccs):.4f}")
    print(f" Top 5% Quantile: {np.quantile(ccs, 0.95):.4f}")
    print(f" Top 1% Quantile: {np.quantile(ccs, 0.99):.4f}")

Downsampling word sequences to TR resolution...
Creating delayed input features...
Pairing TRs with token windows...
Total usable samples: 34009
[subject2] Epoch 1 - Train Loss: 0.9891, Val Loss: 0.9872
[subject2] Epoch 2 - Train Loss: 0.9869, Val Loss: 0.9872
[subject2] Epoch 3 - Train Loss: 0.9869, Val Loss: 0.9872
[subject2] Epoch 4 - Train Loss: 0.9863, Val Loss: 0.9855
[subject2] Epoch 5 - Train Loss: 0.9773, Val Loss: 0.9813
[subject2] Epoch 6 - Train Loss: 0.9678, Val Loss: 0.9805
[subject2] Epoch 7 - Train Loss: 0.9625, Val Loss: 0.9797
[subject2] Epoch 8 - Train Loss: 0.9593, Val Loss: 0.9770
[subject2] Epoch 9 - Train Loss: 0.9557, Val Loss: 0.9756
[subject2] Epoch 10 - Train Loss: 0.9488, Val Loss: 0.9731
[subject2] Epoch 11 - Train Loss: 0.9423, Val Loss: 0.9724
[subject2] Epoch 12 - Train Loss: 0.9377, Val Loss: 0.9715
[subject2] Epoch 13 - Train Loss: 0.9324, Val Loss: 0.9696
[subject2] Epoch 14 - Train Loss: 0.9269, Val Loss: 0.9683
[subject2] Epoch 15 - Train Loss: 0.92

### 1.3. 
Investigate a parameter-efficient approach to fine-tuning this model such as Low-rank adaptation (LORA).
Please refer to the following link as to how to to use LORA with Huggingface https://huggingface.co/docs/diffusers/en/training/lora


In [21]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import BertModel

# === LoRA Config + Model Wrapper ===
class LoRABERTVoxelRegressor(nn.Module):
    def __init__(self, output_dim, r=8, alpha=32, dropout=0.1):
        super().__init__()
        base_model = BertModel.from_pretrained("bert-base-uncased")
        config = LoraConfig(
            r=r,
            lora_alpha=alpha,
            lora_dropout=dropout,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION
        )
        self.bert = get_peft_model(base_model, config)
        self.linear = nn.Linear(self.bert.config.hidden_size, output_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.linear(cls)

# === Training Function (LoRA version) ===
def train_lora_bert(raw_text, fmri_data, subject_name):
    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    dataset = DelayedBERTDataset(raw_text, fmri_data)
    train_size = int(0.7 * len(dataset))
    val_size = int(0.15 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])
    train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32)
    test_loader = DataLoader(test_set, batch_size=32)

    output_dim = dataset[0]["target"].shape[0]
    model = LoRABERTVoxelRegressor(output_dim).to(device)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
    criterion = nn.MSELoss()
    best_val_loss = float("inf")
    patience = 3
    pat_counter = 0

    output_dir = f"output/lora/{subject_name}"
    os.makedirs(output_dir, exist_ok=True)
    best_model_path = os.path.join(output_dir, "bert_lora.pth")

    for epoch in range(20):
        model.train()
        train_loss = 0
        for batch in train_loader:
            inputs = tokenizer(batch["input"], padding=True, truncation=True,
                               return_tensors="pt", max_length=16)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            targets = batch["target"].to(device)
            optimizer.zero_grad()
            preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = tokenizer(batch["input"], padding=True, truncation=True,
                                   return_tensors="pt", max_length=16)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                targets = batch["target"].to(device)
                preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
                val_loss += criterion(preds, targets).item()

        print(f"[{subject_name}] Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            pat_counter = 0
        else:
            pat_counter += 1
            if pat_counter >= patience:
                print(" Early stopping")
                break

    # === Load best and evaluate ===
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs = tokenizer(batch["input"], padding=True, truncation=True,
                               return_tensors="pt", max_length=16)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            targets = batch["target"].cpu().numpy()
            preds = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets)

    pred = np.concatenate(all_preds, axis=0)
    true = np.concatenate(all_targets, axis=0)
    ccs = ridge_corr_pred(pred, pred, true, true, np.ones(true.shape[1]))

    np.save(os.path.join(output_dir, "voxel_cc.npy"), ccs)
    with open(os.path.join(output_dir, "summary_cc.txt"), "w") as f:
        f.write(f"Subject: {subject_name}\\n")
        f.write(f"Mean CC: {np.mean(ccs):.4f}\\n")
        f.write(f"Median CC: {np.median(ccs):.4f}\\n")
        f.write(f"Top 5%: {np.quantile(ccs, 0.95):.4f}\\n")
        f.write(f"Top 1%: {np.quantile(ccs, 0.99):.4f}\\n")

    print(f"Final mean CC: {np.mean(ccs):.4f}")
    return ccs

# === Run for both subjects ===
for name, ydata in subject_map.items():
    print(f"Running LoRA fine-tuning for {name}")
    all_ccs[name + "_lora"] = train_lora_bert(raw_text, ydata, name)

Running LoRA fine-tuning for subject2
Downsampling word sequences to TR resolution...
Creating delayed input features...
Pairing TRs with token windows...
Total usable samples: 34009
[subject2] Epoch 1 - Train Loss: 0.9893, Val Loss: 0.9834
[subject2] Epoch 2 - Train Loss: 0.9880, Val Loss: 0.9830
[subject2] Epoch 3 - Train Loss: 0.9872, Val Loss: 0.9828
[subject2] Epoch 4 - Train Loss: 0.9855, Val Loss: 0.9825
[subject2] Epoch 5 - Train Loss: 0.9828, Val Loss: 0.9837
[subject2] Epoch 6 - Train Loss: 0.9795, Val Loss: 0.9836
[subject2] Epoch 7 - Train Loss: 0.9761, Val Loss: 0.9837
 Early stopping
Final mean CC: 0.3939
Running LoRA fine-tuning for subject3
Downsampling word sequences to TR resolution...
Creating delayed input features...
Pairing TRs with token windows...
Total usable samples: 34786
[subject3] Epoch 1 - Train Loss: 0.9790, Val Loss: 0.9843
[subject3] Epoch 2 - Train Loss: 0.9778, Val Loss: 0.9840
[subject3] Epoch 3 - Train Loss: 0.9769, Val Loss: 0.9836
[subject3] Epoch

In [22]:
for name, ccs in all_ccs.items():
    print(f"== {name} Summary ==")
    print(f" Mean: {np.mean(ccs):.4f}")
    print(f" Median: {np.median(ccs):.4f}")
    print(f" Top 5% Quantile: {np.quantile(ccs, 0.95):.4f}")
    print(f" Top 1% Quantile: {np.quantile(ccs, 0.99):.4f}")

== subject2 Summary ==
 Mean: 0.4253
 Median: 0.4115
 Top 5% Quantile: 0.5123
 Top 1% Quantile: 0.5563
== subject3 Summary ==
 Mean: 0.4301
 Median: 0.4178
 Top 5% Quantile: 0.5162
 Top 1% Quantile: 0.5594
== subject2_lora Summary ==
 Mean: 0.3939
 Median: 0.3936
 Top 5% Quantile: 0.4119
 Top 1% Quantile: 0.4211
== subject3_lora Summary ==
 Mean: 0.3926
 Median: 0.3917
 Top 5% Quantile: 0.4145
 Top 1% Quantile: 0.4252
